# 🐍 Python & SQL Exam

**Instructions:** Work through each question in order. Each section builds on a single scenario — a small online bookshop called **PageTurner**. Read the context carefully before each question.

Good luck! 📚

---
## Part 1 — Python

You are helping build a simple system for PageTurner, an independent online bookshop. Each question adds a new feature to the shop.

### Question 1 — Conditionals

PageTurner applies discounts at checkout based on the customer's cart total:

| Cart Total | Discount |
|---|---|
| Less than $20 | No discount |
| $20 – $49.99 | 5% off |
| $50 – $99.99 | 10% off |
| $100 or more | 15% off |

Write a function `calculate_total(cart_total)` that takes the cart total as a float and **returns a string** in the format:

`"Original: $XX.XX | Discount: X% | Final: $XX.XX"`

**Test your function with:** `32.50`, `75.00`, `120.00`, and `15.00`

In [ ]:
# Question 1 — Your answer here



### Question 2 — Loops & Lists

PageTurner receives a weekly shipment. The shipment is recorded as a list of tuples, where each tuple is `(title, quantity, price_per_unit)`.

```python
shipment = [
    ("The Pragmatic Programmer", 4, 45.99),
    ("Clean Code", 6, 38.50),
    ("Atomic Habits", 10, 22.00),
    ("Deep Work", 3, 29.99),
    ("Dune", 8, 18.75),
]
```

Write code that processes this shipment and prints a summary like this:

```
--- Weekly Shipment Summary ---
The Pragmatic Programmer  →  4 units  @  $45.99  =  $183.96
Clean Code                →  6 units  @  $38.50  =  $231.00
...
------------------------------
Total units received: 31
Total shipment value: $XXX.XX
Most valuable item: The Pragmatic Programmer ($183.96)
```

**Hint:** You'll need to find the most valuable item (highest `quantity × price`) using a loop or built-in functions.

In [ ]:
# Question 2 — Your answer here

shipment = [
    ("The Pragmatic Programmer", 4, 45.99),
    ("Clean Code", 6, 38.50),
    ("Atomic Habits", 10, 22.00),
    ("Deep Work", 3, 29.99),
    ("Dune", 8, 18.75),
]



### Question 3 — Dictionaries

PageTurner stores its current inventory as a dictionary where the key is the book title and the value is the stock count.

```python
inventory = {
    "The Pragmatic Programmer": 4,
    "Clean Code": 6,
    "Atomic Habits": 10,
    "Deep Work": 3,
    "Dune": 8,
    "The Hobbit": 0,
    "1984": 2,
}
```

Write a function `process_order(inventory, order)` where `order` is also a dictionary of `{title: quantity_requested}`. The function should:

1. **Fulfil** items where stock is sufficient (deduct from inventory)
2. **Reject** items where stock is insufficient or the book doesn't exist
3. **Return** a dictionary with two keys: `"fulfilled"` and `"rejected"`, each containing a list of book titles

Test it with this order:
```python
order = {
    "Clean Code": 2,
    "The Hobbit": 1,       # out of stock
    "Atomic Habits": 15,   # not enough stock
    "Deep Work": 1,
    "Harry Potter": 1,     # doesn't exist
}
```

After calling the function, also print the **updated inventory** to confirm stock was deducted correctly.

In [ ]:
# Question 3 — Your answer here

inventory = {
    "The Pragmatic Programmer": 4,
    "Clean Code": 6,
    "Atomic Habits": 10,
    "Deep Work": 3,
    "Dune": 8,
    "The Hobbit": 0,
    "1984": 2,
}

order = {
    "Clean Code": 2,
    "The Hobbit": 1,
    "Atomic Habits": 15,
    "Deep Work": 1,
    "Harry Potter": 1,
}



### Question 4 — Object-Oriented Programming

It's time to bring PageTurner together properly. Design a `Bookshop` system using classes.

#### You need to create two classes:

**`Book`** — represents a single book with:
- Attributes: `title`, `author`, `price`, `stock`
- A method `is_available(quantity)` → returns `True` if stock covers the requested quantity
- A `__str__` method that returns something like: `"Clean Code by Robert Martin — $38.50 (6 in stock)"`

**`Bookshop`** — represents the shop with:
- Attribute: `name`, and a collection of `Book` objects
- `add_book(book)` — adds a Book to the shop
- `search(title)` — returns the Book object if found, or `None`
- `purchase(title, quantity)` — if the book exists and has enough stock, deducts stock and returns the total cost. Otherwise raises an appropriate error message.
- `low_stock_report(threshold=3)` — prints all books with stock at or below the threshold

#### Test it by doing the following:
1. Create a `Bookshop` named `"PageTurner"`
2. Add at least 4 books to it
3. Search for a book and print it
4. Purchase 2 copies of one book and print the cost
5. Try to purchase more copies than are in stock and handle the error gracefully
6. Print the low stock report

In [ ]:
# Question 4 — Your answer here



---
## Part 2 — SQL (SQLite)

The cells below set up a SQLite database for PageTurner's sales records. **Run the setup cell first**, then answer each question by writing a SQL query in the provided cells.

> You can run any query with: `pd.read_sql_query("YOUR SQL HERE", conn)`

In [ ]:
# ── Database Setup — Run this cell first ──────────────────────────────────────
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# ── Schema ─────────────────────────────────────────────────────────────────────

cursor.executescript("""

-- Customers
CREATE TABLE customers (
    customer_id   INTEGER PRIMARY KEY,
    name          TEXT    NOT NULL,
    email         TEXT    UNIQUE NOT NULL,
    city          TEXT,
    joined_date   TEXT    -- ISO format: YYYY-MM-DD
);

-- Books
CREATE TABLE books (
    book_id   INTEGER PRIMARY KEY,
    title     TEXT    NOT NULL,
    author    TEXT    NOT NULL,
    genre     TEXT,
    price     REAL    NOT NULL,
    stock     INTEGER DEFAULT 0
);

-- Orders (one row per order placed)
CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER REFERENCES customers(customer_id),
    order_date  TEXT,
    status      TEXT    -- 'completed', 'pending', 'cancelled'
);

-- Order items (one row per book in an order)
CREATE TABLE order_items (
    item_id   INTEGER PRIMARY KEY,
    order_id  INTEGER REFERENCES orders(order_id),
    book_id   INTEGER REFERENCES books(book_id),
    quantity  INTEGER NOT NULL,
    unit_price REAL   NOT NULL  -- price at time of purchase
);

""")

# ── Seed Data ──────────────────────────────────────────────────────────────────

cursor.executescript("""

INSERT INTO customers VALUES
    (1,  'Alice Tan',     'alice@email.com',   'Singapore', '2022-03-15'),
    (2,  'Ben Lim',       'ben@email.com',     'Kuala Lumpur', '2022-07-22'),
    (3,  'Clara Wong',    'clara@email.com',   'Singapore', '2023-01-10'),
    (4,  'David Ng',      'david@email.com',   'Jakarta',   '2023-04-05'),
    (5,  'Eva Sharma',    'eva@email.com',     'Singapore', '2023-06-18'),
    (6,  'Farid Hassan',  'farid@email.com',   'Kuala Lumpur', '2023-09-30'),
    (7,  'Grace Lee',     'grace@email.com',   'Bangkok',   '2024-01-12'),
    (8,  'Henry Park',    'henry@email.com',   'Seoul',     '2024-02-28');

INSERT INTO books VALUES
    (1,  'The Pragmatic Programmer', 'David Thomas',     'Tech',     45.99, 4),
    (2,  'Clean Code',               'Robert Martin',    'Tech',     38.50, 6),
    (3,  'Atomic Habits',            'James Clear',      'Self-Help',22.00, 10),
    (4,  'Deep Work',                'Cal Newport',      'Self-Help',29.99, 3),
    (5,  'Dune',                     'Frank Herbert',    'Sci-Fi',   18.75, 8),
    (6,  'The Hobbit',               'J.R.R. Tolkien',   'Fantasy',  15.99, 0),
    (7,  '1984',                     'George Orwell',    'Fiction',  12.50, 2),
    (8,  'Thinking, Fast and Slow',  'Daniel Kahneman',  'Self-Help',35.00, 5),
    (9,  'Sapiens',                  'Yuval Noah Harari','History',  28.00, 7),
    (10, 'The Great Gatsby',         'F. Scott Fitzgerald','Fiction',10.99, 3);

INSERT INTO orders VALUES
    (1,  1, '2024-01-05', 'completed'),
    (2,  2, '2024-01-12', 'completed'),
    (3,  1, '2024-02-03', 'completed'),
    (4,  3, '2024-02-14', 'completed'),
    (5,  4, '2024-02-20', 'cancelled'),
    (6,  5, '2024-03-01', 'completed'),
    (7,  2, '2024-03-15', 'completed'),
    (8,  6, '2024-03-22', 'completed'),
    (9,  7, '2024-04-01', 'pending'),
    (10, 1, '2024-04-10', 'completed'),
    (11, 8, '2024-04-18', 'completed'),
    (12, 3, '2024-05-02', 'completed');

INSERT INTO order_items VALUES
    (1,  1,  1, 1, 45.99),  -- Alice buys Pragmatic Programmer
    (2,  1,  3, 2, 22.00),  -- Alice buys 2x Atomic Habits
    (3,  2,  2, 1, 38.50),  -- Ben buys Clean Code
    (4,  2,  5, 1, 18.75),  -- Ben buys Dune
    (5,  3,  4, 1, 29.99),  -- Alice buys Deep Work
    (6,  3,  8, 1, 35.00),  -- Alice buys Thinking Fast and Slow
    (7,  4,  3, 1, 22.00),  -- Clara buys Atomic Habits
    (8,  4,  9, 1, 28.00),  -- Clara buys Sapiens
    (9,  5,  7, 2, 12.50),  -- David (cancelled) — 1984
    (10, 6,  1, 1, 45.99),  -- Eva buys Pragmatic Programmer
    (11, 6,  2, 1, 38.50),  -- Eva buys Clean Code
    (12, 7,  9, 2, 28.00),  -- Ben buys 2x Sapiens
    (13, 7,  5, 1, 18.75),  -- Ben buys Dune
    (14, 8,  3, 3, 22.00),  -- Farid buys 3x Atomic Habits
    (15, 9,  6, 1, 15.99),  -- Grace (pending) — The Hobbit
    (16, 10, 10,1, 10.99),  -- Alice buys Great Gatsby
    (17, 10, 7, 1, 12.50),  -- Alice buys 1984
    (18, 11, 8, 2, 35.00),  -- Henry buys 2x Thinking Fast and Slow
    (19, 11, 4, 1, 29.99),  -- Henry buys Deep Work
    (20, 12, 2, 1, 38.50),  -- Clara buys Clean Code
    (21, 12, 1, 1, 45.99);  -- Clara buys Pragmatic Programmer

""")

conn.commit()
print("✅ Database ready! Tables: customers, books, orders, order_items")

### SQL Question 1 — WHERE & Data Retrieval

**(a)** Retrieve the title, author, genre, and price of all books priced **above $25**, sorted from most expensive to least expensive.

In [ ]:
# SQL Q1a
query = """

"""
pd.read_sql_query(query, conn)

**(b)** Find the **second most expensive** book in the store. Return its title and price.

> **Hint:** Think about ordering and skipping rows.

In [ ]:
# SQL Q1b
query = """

"""
pd.read_sql_query(query, conn)

**(c)** List all **completed** orders placed in **February or March 2024**. Show the order ID, customer ID, and order date.

In [ ]:
# SQL Q1c
query = """

"""
pd.read_sql_query(query, conn)

### SQL Question 2 — Aggregation & Grouping

**(a)** How many books does PageTurner stock in each **genre**? Show the genre and the total number of books (count of titles), sorted by the most books first.

In [ ]:
# SQL Q2a
query = """

"""
pd.read_sql_query(query, conn)

**(b)** What is the **total revenue per genre** from **completed orders only**? Revenue = `quantity × unit_price` per item. Show genre and total revenue, rounded to 2 decimal places, sorted highest first.

> **Hint:** You'll need to join several tables together.

In [ ]:
# SQL Q2b
query = """

"""
pd.read_sql_query(query, conn)

### SQL Question 3 — JOINs

**(a)** List each customer's **name** and the **total number of completed orders** they have placed. Include customers who have placed zero orders too.

> **Hint:** Think about which type of JOIN keeps customers with no orders.

In [ ]:
# SQL Q3a
query = """

"""
pd.read_sql_query(query, conn)

**(b)** Find the **top 3 best-selling books** by total quantity sold (completed orders only). Show the book title, author, and total quantity sold.

In [ ]:
# SQL Q3b
query = """

"""
pd.read_sql_query(query, conn)

**(c) ⭐ Intermediate** — For each customer from **Singapore**, show their name and a comma-separated list of **distinct book titles** they have purchased (from completed orders only).

> **Hint:** Look up `GROUP_CONCAT` in SQLite.

In [ ]:
# SQL Q3c
query = """

"""
pd.read_sql_query(query, conn)